# Data treatment

The main goal of this notebooks is to properly read and cleanup the data from my diffferent bank statements.

## Set-up

In [ ]:
# Basic imports
import csv
import io
from pathlib import Path

import pandas as pd

# Financial sim imports

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option(
    "display.max_colwidth", None
)  # opcional: no recorta textos largos como descripciones

In [ ]:
# Data path
path_project_root = next(
    path_candidate
    for path_candidate in [Path.cwd(), *Path.cwd().parents]
    if (path_candidate / "data" / "raw").exists()
)
path_raw_data = path_project_root / "data" / "raw"

## Transactional data imports

In [ ]:
# Importing BBVA data, easy, just skip and cut left
df_bbva_raw = pd.read_excel(path_raw_data / "bbva_full_data.xlsx", header=4).iloc[:, 1:]
df_bbva_raw.head()

In [ ]:
# Importing revolut data, a little more complex as it has multiple wallets
list_revolut_lines = (
    (path_raw_data / "revolut_full_data.csv").read_text(encoding="utf-8").splitlines()
)


def get_first_cell(line: str) -> str:
    return next(csv.reader([line]), [""])[0].strip() if line else ""


# Each wallet's transaction table starts right after an "Extracto de transacción" marker
list_marker_indices = [
    index_line
    for index_line, line in enumerate(list_revolut_lines)
    if get_first_cell(line) == "Extracto de transacción"
]

dict_column_names = {
    "Fecha": "date",
    "Descripción": "description",
    "Categoría": "category",
    "Entradas/salidas de dinero": "amount",
    "Saldo": "balance",
    "Impuestos retenidos": "taxes_withheld",
    "Otros impuestos": "other_taxes",
    "Comisiones": "fees",
}
list_money_columns = ["amount", "balance", "taxes_withheld", "other_taxes", "fees"]


def rename_revolut_column(column_name: str) -> str:
    base_column_name = column_name.removesuffix(".1")
    suffix = "_eur" if column_name.endswith(".1") else ""
    return dict_column_names.get(base_column_name, base_column_name) + suffix


def parse_money(series_money: pd.Series) -> pd.Series:
    series_clean = (
        series_money.astype(str)
        .str.replace("−", "-", regex=False)
        .str.replace(r"[^\d,.\-]", "", regex=True)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    return pd.to_numeric(series_clean, errors="coerce")


list_df_wallets = []
for index_marker in list_marker_indices:
    wallet_name = next(
        get_first_cell(list_revolut_lines[index_line])
        for index_line in range(index_marker - 1, -1, -1)
        if get_first_cell(list_revolut_lines[index_line])
    )
    index_header = index_marker + 1
    index_total = next(
        (
            index_line
            for index_line in range(index_header + 1, len(list_revolut_lines))
            if get_first_cell(list_revolut_lines[index_line]) == "Total"
        ),
        len(list_revolut_lines),
    )

    df_wallet = pd.read_csv(
        io.StringIO("\n".join(list_revolut_lines[index_header:index_total]))
    )
    df_wallet = df_wallet.loc[:, ~df_wallet.columns.str.startswith("Unnamed")]
    df_wallet = df_wallet.rename(columns=rename_revolut_column)

    for money_column in list_money_columns:
        if f"{money_column}_eur" not in df_wallet.columns:
            df_wallet[f"{money_column}_eur"] = df_wallet[money_column]
        df_wallet[money_column] = parse_money(df_wallet[money_column])
        df_wallet[f"{money_column}_eur"] = parse_money(df_wallet[f"{money_column}_eur"])

    df_wallet["wallet"] = wallet_name
    list_df_wallets.append(df_wallet)

df_revolut_raw = pd.concat(list_df_wallets, ignore_index=True)
df_revolut_raw["currency"] = df_revolut_raw["wallet"].str.extract(r"\((\w{3})\)")

df_revolut_raw.head()

In [ ]:
df_tr_raw = pd.read_csv(path_raw_data / "tr_full_data.csv")
df_tr_raw.head()

## Data unification

Single transactions dataframe with every field available across BBVA, Revolut and Trade Republic. Columns that a source does not provide are left empty. Money follows the cash flow sign: income is positive and spending, fees and taxes are negative.

In [ ]:
# Unified schema: column order and data type for every column of the final dataframe
dict_unified_schema = {
    # Origin
    "account": pd.CategoricalDtype(["bbva", "revolut", "trade_republic"]),
    "sub_account": "string",
    "transaction_id": "string",
    # Dates
    "date": "datetime64[ns]",
    "value_date": "datetime64[ns]",
    "timestamp": "datetime64[ns, UTC]",
    # Classification and descriptions
    "transaction_type": "string",
    "category": "string",
    "description": "string",
    "details": "string",
    # Counterparty
    "counterparty_name": "string",
    "counterparty_iban": "string",
    "payment_reference": "string",
    "mcc_code": "Int64",
    # Investment
    "asset_class": "string",
    "asset_name": "string",
    "asset_isin": "string",
    "shares": "Float64",
    "price": "Float64",
    # Money (costs such as fees and taxes are negative)
    "amount": "Float64",
    "currency": "string",
    "amount_eur": "Float64",
    "fee": "Float64",
    "fee_eur": "Float64",
    "tax": "Float64",
    "tax_eur": "Float64",
    "original_amount": "Float64",
    "original_currency": "string",
    "fx_rate": "Float64",
    "balance": "Float64",
    "balance_eur": "Float64",
}

In [ ]:
# BBVA: single EUR account, dates in dd/mm/yyyy
df_bbva = pd.DataFrame(
    {
        "account": "bbva",
        "date": pd.to_datetime(df_bbva_raw["Fecha"], format="%d/%m/%Y"),
        "value_date": pd.to_datetime(df_bbva_raw["F.Valor"], format="%d/%m/%Y"),
        "transaction_type": df_bbva_raw["Movimiento"],
        "description": df_bbva_raw["Concepto"].str.split().str.join(" "),
        "details": df_bbva_raw["Observaciones"].str.split().str.join(" "),
        "amount": df_bbva_raw["Importe"],
        "currency": df_bbva_raw["Divisa"],
        "amount_eur": df_bbva_raw["Importe"],
        "balance": df_bbva_raw["Disponible"],
        "balance_eur": df_bbva_raw["Disponible"],
    }
)
df_bbva.head()

In [ ]:
# Revolut: one row per wallet movement, dates like "8 jul 2025" with Spanish month abbreviations
dict_spanish_months = {
    "ene": "01",
    "feb": "02",
    "mar": "03",
    "abr": "04",
    "may": "05",
    "jun": "06",
    "jul": "07",
    "ago": "08",
    "sept": "09",
    "sep": "09",
    "oct": "10",
    "nov": "11",
    "dic": "12",
}


def parse_spanish_date(series_date: pd.Series) -> pd.Series:
    series_parts = series_date.str.split(expand=True)
    series_iso = (
        series_parts[2]
        + "-"
        + series_parts[1].map(dict_spanish_months)
        + "-"
        + series_parts[0].str.zfill(2)
    )
    return pd.to_datetime(series_iso, format="%Y-%m-%d")


df_revolut = pd.DataFrame(
    {
        "account": "revolut",
        "sub_account": df_revolut_raw["wallet"],
        "date": parse_spanish_date(df_revolut_raw["date"]),
        "category": df_revolut_raw["category"],
        "description": df_revolut_raw["description"],
        "amount": df_revolut_raw["amount"],
        "currency": df_revolut_raw["currency"],
        "amount_eur": df_revolut_raw["amount_eur"],
        # Revolut reports fees and taxes as positive costs, flip them to match the cash flow sign
        "fee": 0 - df_revolut_raw["fees"],
        "fee_eur": 0 - df_revolut_raw["fees_eur"],
        "tax": 0 - (df_revolut_raw["taxes_withheld"] + df_revolut_raw["other_taxes"]),
        "tax_eur": 0
        - (df_revolut_raw["taxes_withheld_eur"] + df_revolut_raw["other_taxes_eur"]),
        "balance": df_revolut_raw["balance"],
        "balance_eur": df_revolut_raw["balance_eur"],
    }
)
df_revolut.head()

In [ ]:
# Trade Republic: everything is already in EUR, account_type is always DEFAULT so it is dropped
# "name" holds the asset for trades and the merchant / counterparty for cash movements
is_asset = df_tr_raw["asset_class"].notna()

df_tr = pd.DataFrame(
    {
        "account": "trade_republic",
        "sub_account": df_tr_raw["category"].str.lower(),
        "transaction_id": df_tr_raw["transaction_id"],
        "date": pd.to_datetime(df_tr_raw["date"], format="%Y-%m-%d"),
        "timestamp": pd.to_datetime(df_tr_raw["datetime"], format="ISO8601", utc=True),
        "transaction_type": df_tr_raw["type"],
        "description": df_tr_raw["description"],
        "counterparty_name": df_tr_raw["counterparty_name"].fillna(
            df_tr_raw["name"].where(~is_asset)
        ),
        "counterparty_iban": df_tr_raw["counterparty_iban"],
        "payment_reference": df_tr_raw["payment_reference"],
        "mcc_code": df_tr_raw["mcc_code"],
        "asset_class": df_tr_raw["asset_class"],
        "asset_name": df_tr_raw["name"].where(is_asset),
        "asset_isin": df_tr_raw["symbol"],
        "shares": df_tr_raw["shares"],
        "price": df_tr_raw["price"],
        "amount": df_tr_raw["amount"],
        "currency": df_tr_raw["currency"],
        "amount_eur": df_tr_raw["amount"],
        "fee": df_tr_raw["fee"],
        "fee_eur": df_tr_raw["fee"],
        "tax": df_tr_raw["tax"],
        "tax_eur": df_tr_raw["tax"],
        "original_amount": df_tr_raw["original_amount"],
        "original_currency": df_tr_raw["original_currency"],
        "fx_rate": df_tr_raw["fx_rate"],
    }
)
df_tr.head()

In [ ]:
# Stack all accounts, add the missing columns, enforce order and types, sort chronologically
df_transactions = (
    pd.concat([df_bbva, df_revolut, df_tr], ignore_index=True)
    .reindex(columns=list(dict_unified_schema))
    .astype(dict_unified_schema)
    .sort_values(["date", "timestamp"], kind="stable", ignore_index=True)
)
df_transactions.info()

In [ ]:
df_transactions.groupby("account", observed=True).agg(
    n_transactions=("date", "size"),
    first_date=("date", "min"),
    last_date=("date", "max"),
    total_amount_eur=("amount_eur", "sum"),
)

## Data spliting

In [ ]:
df_tr_clean = df_transactions[df_transactions["account"] == "trade_republic"]
df_tr_stock_movement = df_tr_clean[df_tr_clean["sub_account"] == "trading"]
display(df_tr_stock_movement)

In [ ]:
from functools import lru_cache

import yfinance as yf

# Posiciones abiertas hoy (Defence, Uranium y D-Wave quedan a 0 y se descartan)
series_shares_today = df_tr_stock_movement.groupby("asset_isin")["shares"].sum()
series_shares_today = series_shares_today[series_shares_today.abs() > 1e-6]

df_tr_sorted = df_tr_stock_movement.sort_values("timestamp")
series_asset_name = df_tr_sorted.groupby("asset_isin")["asset_name"].last()
series_last_trade_price_eur = df_tr_sorted.groupby("asset_isin")["price"].last()

dict_isin_to_ticker_override = {
    "IE00B4L5Y983": "EUNL.DE",  # Core MSCI World, Xetra
    "IE00BKM4GZ66": "IS3N.DE",  # Core MSCI EM IMI, Xetra
    "ES0113211835": "BBVA.MC",
    "ES0113900J37": "SAN.MC",
}


@lru_cache
def get_eur_per_unit(currency: str) -> float:
    if currency == "EUR":
        return 1.0
    if currency == "GBp":  # cotizaciones de Londres en peniques
        return get_eur_per_unit("GBP") / 100
    rate_currency_per_eur = (
        yf.Ticker(f"EUR{currency}=X").history(period="5d")["Close"].iloc[-1]
    )
    return 1 / rate_currency_per_eur


list_rows_valuation = []
for asset_isin, shares in series_shares_today.items():
    ticker = yf.Ticker(dict_isin_to_ticker_override.get(asset_isin, asset_isin))
    df_price_history = ticker.history(period="5d")
    if df_price_history.empty:
        print(
            f"Sin precio para {asset_isin} ({series_asset_name[asset_isin]}): añade un override"
        )
        continue
    currency = ticker.fast_info["currency"]
    last_price_eur = df_price_history["Close"].iloc[-1] * get_eur_per_unit(currency)
    list_rows_valuation.append(
        {
            "asset_isin": asset_isin,
            "asset_name": series_asset_name[asset_isin],
            "ticker": ticker.ticker,
            "listing_currency": currency,
            "shares": shares,
            "last_price_eur": last_price_eur,
            "last_trade_price_eur": series_last_trade_price_eur[asset_isin],
            "value_eur": shares * last_price_eur,
        }
    )

df_valuation_today = pd.DataFrame(list_rows_valuation)
df_valuation_today["price_vs_last_trade"] = (
    df_valuation_today["last_price_eur"] / df_valuation_today["last_trade_price_eur"]
)
display(df_valuation_today.sort_values("value_eur", ascending=False))
print(f"Valor cartera TR: {df_valuation_today['value_eur'].sum():,.2f} €")